# 16 — Expanded LIBERO-PRO analysis

Zero-GPU analysis of the completed workers 0–5 experiment (`pro-16suite-k5-steps34-v1`). It snapshots the relevant Supabase rows, validates all six shards and paired identities, recreates the success/refinement summaries, and tests whether observed-arm consecutive uncertainty contraction predicts a matched failure-to-success correction.

The contraction analysis uses `pnp_action_vectors.u_iter`; full `a_hat` vectors are neither loaded nor required.

## 1. Setup

In [ ]:
EXTRAS = 'analysis'
SETUP_ENV = False
import urllib.request
exec(urllib.request.urlopen('https://raw.githubusercontent.com/ArjunS07/cs159-sp26/main/pnp-vla/scripts/colab_bootstrap.py').read().decode())

## 2. Refresh the Supabase snapshot and run the validated report

In [ ]:
from analysis.run_analysis import main
from pnp.experiments import PRO_EXPANDED_EXPERIMENT

OUTPUT_ROOT = 'analysis_outputs'
main(['pro-expanded', '--experiment', PRO_EXPANDED_EXPERIMENT,
      '--output-root', OUTPUT_ROOT, '--refresh'])

## 3. Primary rollout and contraction results

Positive contraction scores mean disagreement decreases over the four consecutive perturbation pairs. `corrected` means the observed/no-op rollout failed and the paired refine-last rollout succeeded. The primary rank score is normalized within suite to reduce suite-scale confounding.

In [ ]:
import json
import pandas as pd
from analysis.snapshot import latest_snapshot

snapshot_path = latest_snapshot(OUTPUT_ROOT, PRO_EXPANDED_EXPERIMENT)
tables = snapshot_path / 'tables'
print('snapshot:', snapshot_path)
validation = json.loads((snapshot_path / 'validation.json').read_text())
print({key: validation.get(key) for key in (
    'n_snapshot_rollouts', 'n_rollouts', 'n_ignored_rollouts',
    'n_identities', 'n_suites', 'control_complete')})
for warning in validation.get('warnings', []): print('WARNING:', warning)
display(pd.read_csv(tables / 'pro_success_by_suite.csv'))
display(pd.read_csv(tables / 'pro_paired_by_suite.csv'))
summary = pd.read_csv(tables / 'expanded_contraction_summary.csv')
display(summary[summary.metric.isin([
    'contraction_within_suite_rank', 'contraction_normalized_slope'
])])
display(pd.read_csv(tables / 'expanded_contraction_quartiles.csv'))

print('Uncertainty as a failure detector')
display(pd.read_csv(tables / 'pro_detector_summary.csv'))
display(pd.read_csv(tables / 'pro_detector_by_suite.csv'))

print('Paired refinement effect by observed-uncertainty decile')
display(pd.read_csv(tables / 'expanded_refinement_effect_by_uncertainty_bin.csv'))

print('Best exploratory uncertainty windows (selected on these same outcomes)')
window_sweep = pd.read_csv(tables / 'expanded_uncertainty_window_sweep.csv')
display(window_sweep.sort_values(['delta_pp', 'n_refined'], ascending=False).head(20))

## 4. Figures

In [ ]:
from IPython.display import Image, display
for name in ('expanded_pro_success_by_suite',
             'expanded_pro_paired_transitions',
             'expanded_contraction_profiles',
             'expanded_contraction_quartiles',
             'expanded_uncertainty_failure_auc',
             'expanded_refinement_effect_by_uncertainty',
             'expanded_uncertainty_window_sweep'):
    print(name)
    display(Image(filename=str(snapshot_path / 'figures' / f'{name}.png')))